<a href="https://colab.research.google.com/github/tpedCode/P07/blob/feature%2Feda/notebooks/1_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Implémentez un modèle de scoring**

### Objectif :
- prédire la probabilité de défaut d’un client
- optimiser un seuil métier (coût FN > FP)
- préparer un modèle pour une API

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import os

In [ ]:
# Base path (Google Drive)
BASE_PATH = "/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/data/"

# Chemins de données
RAW_PATH = os.path.join(BASE_PATH, "raw/")
PROCESSED_PATH = os.path.join(BASE_PATH, "processed/")

RAW_PATH, PROCESSED_PATH

('/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/data/raw/',
 '/content/drive/MyDrive/Pro/Data_Scientist_by_Openclassrooms_en_cours/P07/data/processed/')

## **ANALYSE EXPLORATOIRE**

In [ ]:
def analyze_dataframe(df, name="DataFrame", top_n=20):
    print(f"===== {name.upper()} =====")

    print("\nShape:", df.shape)

    display(df.head())

    # Target si disponible
    if "TARGET" in df.columns:
        print("\nTarget distribution:")
        print(df["TARGET"].value_counts(normalize=True))

    # ===== TABLEAU RÉCAP =====
    zero_pct = pd.Series(index=df.columns, dtype=float)
    num_cols = df.select_dtypes(include=np.number).columns
    zero_pct[num_cols] = (df[num_cols] == 0).mean() * 100
    zero_pct = zero_pct.fillna(0)

    summary_df = pd.DataFrame({
        "missing_%": (df.isnull().mean() * 100).round(2),
        "zero_%": zero_pct.round(2),
        "dtype": df.dtypes
    }).sort_values(by="missing_%", ascending=False)

    print(f"\nColumn summary (top {top_n}):")
    display(summary_df.head(top_n))

    # ===== NUMERICAL SUMMARY =====
    print("\nNumerical summary (transposed):")
    display(df.describe().T)

    # ===== CATEGORICAL FEATURES =====
    print("\nCategorical features:")
    display(
        df.select_dtypes(include="object")
        .nunique()
        .sort_values(ascending=False)
        .head(top_n)
    )

In [ ]:
# APPLICATION TRAIN

df_train = pd.read_csv(os.path.join(RAW_PATH, "application_train.csv"))

analyze_dataframe(df_train, name="application_train")


===== APPLICATION_TRAIN =====

Shape: (307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0



Target distribution:
TARGET
0    0.919271
1    0.080729
Name: proportion, dtype: float64

Column summary (top 20):


,missing_%,zero_%,dtype
COMMONAREA_AVG,69.87,2.75,float64
COMMONAREA_MODE,69.87,3.15,float64
COMMONAREA_MEDI,69.87,2.83,float64
NONLIVINGAPARTMENTS_MEDI,69.43,18.24,float64
NONLIVINGAPARTMENTS_MODE,69.43,19.27,float64
NONLIVINGAPARTMENTS_AVG,69.43,17.74,float64
FONDKAPREMONT_MODE,68.39,0.00,object
LIVINGAPARTMENTS_AVG,68.35,0.14,float64
LIVINGAPARTMENTS_MEDI,68.35,0.14,float64
LIVINGAPARTMENTS_MODE,68.35,0.17,float64



Numerical summary (transposed):


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,307511.0,278180.518577,102790.175348,100002.0,189145.5,278202.0,367142.5,456255.0
TARGET,307511.0,0.080729,0.272419,0.0,0.0,0.0,0.0,1.0
CNT_CHILDREN,307511.0,0.417052,0.722121,0.0,0.0,0.0,1.0,19.0
AMT_INCOME_TOTAL,307511.0,168797.919297,237123.146279,25650.0,112500.0,147150.0,202500.0,117000000.0
AMT_CREDIT,307511.0,599025.999706,402490.776996,45000.0,270000.0,513531.0,808650.0,4050000.0
...,...,...,...,...,...,...,...,...
AMT_REQ_CREDIT_BUREAU_DAY,265992.0,0.007000,0.110757,0.0,0.0,0.0,0.0,9.0
AMT_REQ_CREDIT_BUREAU_WEEK,265992.0,0.034362,0.204685,0.0,0.0,0.0,0.0,8.0
AMT_REQ_CREDIT_BUREAU_MON,265992.0,0.267395,0.916002,0.0,0.0,0.0,0.0,27.0
AMT_REQ_CREDIT_BUREAU_QRT,265992.0,0.265474,0.794056,0.0,0.0,0.0,0.0,261.0



Categorical features:


,0
ORGANIZATION_TYPE,58
OCCUPATION_TYPE,18
NAME_INCOME_TYPE,8
NAME_TYPE_SUITE,7
WALLSMATERIAL_MODE,7
WEEKDAY_APPR_PROCESS_START,7
NAME_FAMILY_STATUS,6
NAME_HOUSING_TYPE,6
NAME_EDUCATION_TYPE,5
FONDKAPREMONT_MODE,4


In [ ]:
# APPLICATION TEST

df_test = pd.read_csv(os.path.join(RAW_PATH, "application_test.csv"))

print("Shape:", df_test.shape)

display(df_test.head())

print("\nMissing values (top 10):")
print(df_test.isnull().mean().sort_values(ascending=False).head(10))

print("\nData types:")
df_test.info()

Shape: (48744, 121)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN



Missing values (top 10):
COMMONAREA_AVG              0.687161
COMMONAREA_MEDI             0.687161
COMMONAREA_MODE             0.687161
NONLIVINGAPARTMENTS_AVG     0.684125
NONLIVINGAPARTMENTS_MEDI    0.684125
NONLIVINGAPARTMENTS_MODE    0.684125
FONDKAPREMONT_MODE          0.672842
LIVINGAPARTMENTS_MEDI       0.672493
LIVINGAPARTMENTS_AVG        0.672493
LIVINGAPARTMENTS_MODE       0.672493
dtype: float64

Data types:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48744 entries, 0 to 48743
Columns: 121 entries, SK_ID_CURR to AMT_REQ_CREDIT_BUREAU_YEAR
dtypes: float64(65), int64(40), object(16)
memory usage: 45.0+ MB


In [ ]:
# BUREAU

bureau = pd.read_csv(os.path.join(RAW_PATH, "bureau.csv"))

print("Shape:", bureau.shape)

display(bureau.head())

print("\nMissing values (top 10):")
print(bureau.isnull().mean().sort_values(ascending=False).head(10))

print("\nKey columns:")
display(bureau[["SK_ID_CURR", "SK_ID_BUREAU"]].head())

print("\nCredit status distribution:")
print(bureau["CREDIT_ACTIVE"].value_counts())

Shape: (1716428, 17)


,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,464323.5,NaN,NaN,0.0,Consumer credit,-16,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,90000.0,NaN,NaN,0.0,Credit card,-16,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,2700000.0,NaN,NaN,0.0,Consumer credit,-21,NaN



Missing values (top 10):
AMT_ANNUITY               0.714735
AMT_CREDIT_MAX_OVERDUE    0.655133
DAYS_ENDDATE_FACT         0.369170
AMT_CREDIT_SUM_LIMIT      0.344774
AMT_CREDIT_SUM_DEBT       0.150119
DAYS_CREDIT_ENDDATE       0.061496
AMT_CREDIT_SUM            0.000008
SK_ID_CURR                0.000000
SK_ID_BUREAU              0.000000
CREDIT_DAY_OVERDUE        0.000000
dtype: float64

Key columns:


,SK_ID_CURR,SK_ID_BUREAU
0,215354,5714462
1,215354,5714463
2,215354,5714464
3,215354,5714465
4,215354,5714466



Credit status distribution:
CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64


In [ ]:
# BUREAU BALANCE

bureau_balance = pd.read_csv(os.path.join(RAW_PATH, "bureau_balance.csv"))

print("Shape:", bureau_balance.shape)

display(bureau_balance.head())

print("\nStatus distribution:")
print(bureau_balance["STATUS"].value_counts())

print("\nTime range (months):")
display(bureau_balance["MONTHS_BALANCE"].describe())


Shape: (27299925, 3)


,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C
2,5715448,-2,C
3,5715448,-3,C
4,5715448,-4,C



Status distribution:
STATUS
C    13646993
0     7499507
X     5810482
1      242347
5       62406
2       23419
3        8924
4        5847
Name: count, dtype: int64

Time range (months):


,MONTHS_BALANCE
count,2.729992e+07
mean,-3.074169e+01
std,2.386451e+01
min,-9.600000e+01
25%,-4.600000e+01
50%,-2.500000e+01
75%,-1.100000e+01
max,0.000000e+00


In [ ]:
# PREVIOUS APPLICATION

prev_app = pd.read_csv(os.path.join(RAW_PATH, "previous_application.csv"))

print("Shape:", prev_app.shape)

display(prev_app.head())

print("\nApplication status:")
print(prev_app["NAME_CONTRACT_STATUS"].value_counts())

print("\nMissing values (top 10):")
print(prev_app.isnull().mean().sort_values(ascending=False).head(10))


Shape: (1670214, 37)


,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN



Application status:
NAME_CONTRACT_STATUS
Approved        1036781
Canceled         316319
Refused          290678
Unused offer      26436
Name: count, dtype: int64

Missing values (top 10):
RATE_INTEREST_PRIVILEGED     0.996437
RATE_INTEREST_PRIMARY        0.996437
AMT_DOWN_PAYMENT             0.536365
RATE_DOWN_PAYMENT            0.536365
NAME_TYPE_SUITE              0.491198
DAYS_TERMINATION             0.402981
DAYS_FIRST_DRAWING           0.402981
DAYS_FIRST_DUE               0.402981
DAYS_LAST_DUE_1ST_VERSION    0.402981
DAYS_LAST_DUE                0.402981
dtype: float64


In [ ]:
# POS CASH BALANCE

pos_cash = pd.read_csv(os.path.join(RAW_PATH, "POS_CASH_balance.csv"))

print("Shape:", pos_cash.shape)

display(pos_cash.head())

print("\nDPD stats:")
display(pos_cash["SK_DPD"].describe())

print("\nMissing values:")
print(pos_cash.isnull().mean().sort_values(ascending=False).head(10))


Shape: (10001358, 8)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0
2,1784872,397406,-32,12.0,9.0,Active,0,0
3,1903291,269225,-35,48.0,42.0,Active,0,0
4,2341044,334279,-35,36.0,35.0,Active,0,0



DPD stats:


,SK_DPD
count,1.000136e+07
mean,1.160693e+01
std,1.327140e+02
min,0.000000e+00
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
max,4.231000e+03



Missing values:
CNT_INSTALMENT_FUTURE    0.002608
CNT_INSTALMENT           0.002607
SK_ID_CURR               0.000000
SK_ID_PREV               0.000000
MONTHS_BALANCE           0.000000
NAME_CONTRACT_STATUS     0.000000
SK_DPD                   0.000000
SK_DPD_DEF               0.000000
dtype: float64


In [ ]:
# CREDIT CARD BALANCE

cc_balance = pd.read_csv(os.path.join(RAW_PATH, "credit_card_balance.csv"))

print("Shape:", cc_balance.shape)

display(cc_balance.head())

print("\nBalance stats:")
display(cc_balance["AMT_BALANCE"].describe())

print("\nMissing values:")
print(cc_balance.isnull().mean().sort_values(ascending=False).head(10))


Shape: (3840312, 23)


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0
2,1740877,371185,-7,31815.225,450000,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0,0.0,0.0,30.0,Active,0,0
3,1389973,337855,-4,236572.110,225000,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1,0.0,0.0,10.0,Active,0,0
4,1891521,126868,-1,453919.455,450000,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1,0.0,1.0,101.0,Active,0,0



Balance stats:


,AMT_BALANCE
count,3.840312e+06
mean,5.830016e+04
std,1.063070e+05
min,-4.202502e+05
25%,0.000000e+00
50%,0.000000e+00
75%,8.904669e+04
max,1.505902e+06



Missing values:
AMT_PAYMENT_CURRENT           0.199981
CNT_DRAWINGS_POS_CURRENT      0.195249
AMT_DRAWINGS_ATM_CURRENT      0.195249
CNT_DRAWINGS_ATM_CURRENT      0.195249
AMT_DRAWINGS_POS_CURRENT      0.195249
AMT_DRAWINGS_OTHER_CURRENT    0.195249
CNT_DRAWINGS_OTHER_CURRENT    0.195249
CNT_INSTALMENT_MATURE_CUM     0.079482
AMT_INST_MIN_REGULARITY       0.079482
AMT_DRAWINGS_CURRENT          0.000000
dtype: float64


In [ ]:
# INSTALLMENTS PAYMENTS

installments = pd.read_csv(os.path.join(RAW_PATH, "installments_payments.csv"))

print("Shape:", installments.shape)

display(installments.head())

print("\nDelay analysis:")
installments["DELAY"] = installments["DAYS_ENTRY_PAYMENT"] - installments["DAYS_INSTALMENT"]
display(installments["DELAY"].describe())

print("\nMissing values:")
print(installments.isnull().mean().sort_values(ascending=False).head(10))

Shape: (13605401, 8)


,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525
2,2085231,193053,2.0,1,-63.0,-63.0,25425.000,25425.000
3,2452527,199697,1.0,3,-2418.0,-2426.0,24350.130,24350.130
4,2714724,167756,1.0,2,-1383.0,-1366.0,2165.040,2160.585



Delay analysis:


,DELAY
count,1.360250e+07
mean,-8.787405e+00
std,2.553990e+01
min,-3.189000e+03
25%,-1.400000e+01
50%,-6.000000e+00
75%,0.000000e+00
max,2.884000e+03



Missing values:
AMT_PAYMENT               0.000214
DAYS_ENTRY_PAYMENT        0.000214
DELAY                     0.000214
SK_ID_PREV                0.000000
SK_ID_CURR                0.000000
DAYS_INSTALMENT           0.000000
NUM_INSTALMENT_NUMBER     0.000000
NUM_INSTALMENT_VERSION    0.000000
AMT_INSTALMENT            0.000000
dtype: float64


In [ ]:
# COLUMNS DESCRIPTION

desc = pd.read_csv(
    os.path.join(RAW_PATH, "HomeCredit_columns_description.csv"),
    encoding="latin1"
)

print("Shape:", desc.shape)

display(desc.head())

print("\nColonnes disponibles :")
print(desc.columns.tolist())

print("\nExemple de descriptions :")
display(desc.sample(5))


Shape: (219, 5)


,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN



Colonnes disponibles :
['Unnamed: 0', 'Table', 'Row', 'Description', 'Special']

Exemple de descriptions :


,Unnamed: 0,Table,Row,Description,Special
38,41,application_{train|test}.csv,REG_CITY_NOT_WORK_CITY,Flag if client's permanent address does not ma...,NaN
26,29,application_{train|test}.csv,FLAG_PHONE,"Did client provide home phone (1=YES, 0=NO)",NaN
12,15,application_{train|test}.csv,NAME_INCOME_TYPE,"Clients income type (businessman, working, mat...",NaN
186,189,previous_application.csv,RATE_DOWN_PAYMENT,Down payment rate normalized on previous credit,normalized
41,44,application_{train|test}.csv,EXT_SOURCE_1,Normalized score from external data source,normalized


In [ ]:
# SAMPLE SUBMISSION

sample = pd.read_csv(os.path.join(RAW_PATH, "sample_submission.csv"))

print("Shape:", sample.shape)

display(sample.head())

Shape: (48744, 2)


,SK_ID_CURR,TARGET
0,100001,0.5
1,100005,0.5
2,100013,0.5
3,100028,0.5
4,100038,0.5


### Analyse globale des données

- Le dataset principal `application_train` contient **307 511 observations et 122 variables**
- Le dataset `application_test` contient **48 744 observations et 121 variables**
- La variable cible (`TARGET`) n’est présente que dans le dataset train
- Les données sont **fortement déséquilibrées** :
  - ~91.9% de non défaut (TARGET = 0)
  - ~8.1% de défaut (TARGET = 1)
- Le problème est donc une **classification binaire déséquilibrée**
- Les données contiennent :
  - variables numériques (majoritaires)
  - variables catégorielles
  - un grand nombre de valeurs manquantes
- Certaines variables ont des valeurs extrêmes (ex : AMT_INCOME_TOTAL très élevé)


### Analyse unitaire — application_train

- Dataset principal utilisé pour l’entraînement du modèle
- Très grand nombre de variables (122)
- Forte présence de valeurs manquantes :
  - certaines variables ont plus de **65% de valeurs manquantes**
- Variables catégorielles avec différentes cardinalités (jusqu’à 58 modalités)
- Variables temporelles exprimées en jours négatifs (ex : DAYS_BIRTH, DAYS_EMPLOYED)
- Dataset structuré mais nécessitant :
  - nettoyage
  - imputation
  - encodage


### Analyse unitaire — application_test

- Structure similaire à `application_train`
- Absence de la variable cible (`TARGET`)
- Même problématique de valeurs manquantes
- Utilisé uniquement pour la prédiction finale


### Analyse unitaire — bureau

- Historique des crédits externes pour chaque client
- Relation :
  - SK_ID_CURR → client
  - SK_ID_BUREAU → crédit
- Plusieurs crédits par client (relation 1-N)
- Données comportant des valeurs manquantes importantes
- Informations utiles :
  - statut du crédit (actif, clôturé)
  - montants
  - dettes


### Analyse unitaire — bureau_balance

- Données temporelles associées aux crédits bureau
- Très volumineux (~27 millions de lignes)
- Historique mensuel des comportements
- Variables essentielles :
  - STATUS → comportement de paiement
  - MONTHS_BALANCE → axe temporel
- Nécessite une agrégation pour être exploitable


### Analyse unitaire — previous_application

- Historique des demandes de crédit
- Plusieurs lignes par client
- Variables importantes :
  - statut de la demande (accepté, refusé)
- Très forte présence de valeurs manquantes (>50% sur certaines colonnes)


### Analyse unitaire — POS_CASH_balance

- Historique des crédits POS (points de vente)
- Données temporelles
- Variables importantes :
  - SK_DPD → retard de paiement
- Majorité de valeurs nulles → faible activité de défaut visible directement


### Analyse unitaire — credit_card_balance

- Informations sur les cartes de crédit
- Volume important (~3.8M lignes)
- Données financières détaillées (balances, limites, tirages)
- Présence de valeurs manquantes (~20% pour certaines variables)


### Analyse unitaire — installments_payments

- Historique des paiements d’échéances
- Très volumineux (~13M lignes)
- Création d’une variable intéressante :
  - DELAY = retard de paiement
- Distribution des retards :
  - majoritairement nuls ou négatifs
  - quelques retards très élevés (outliers)


### Analyse unitaire — description des colonnes

- Fournit la définition des variables
- Utile pour :
  - interpréter les features
  - comprendre le contexte métier
- Certaines variables sont normalisées (EXT_SOURCE)


### Analyse unitaire — sample_submission

- Format attendu pour la soumission finale
- Ne contient pas d’information utile pour le modèle


### Conclusion générale

- Le dataset est composé de plusieurs tables relationnelles
- Le problème est un cas classique de **scoring crédit**
- Les principales difficultés sont :
  - données déséquilibrées
  - très grand nombre de variables
  - nombreuses valeurs manquantes
  - structure multi-tables nécessitant des jointures

- Les prochaine étapes seront :
  - nettoyage des données
  - agrégation des tables secondaires
  - feature engineering
  - modélisation avec suivi MLflow